# Single-Vg chemical-gas results and training

The default run reads the archived results: 24 cropped Vg channels
(indices 0, 4, ..., 92), 24 parameter sets per channel, and seed 0: **576 completed runs**.
Saved tables and prediction artifacts support result inspection without loading raw sensor data.

Training and validation use RH 0%; held-out reporting includes RH 0%, 30%, and 50%.
Each model uses one Vg channel across the full 42-point time window. Candidate selection
uses maximum validation accuracy, with the earliest candidate winning an exact tie.
Each checkpoint uses the earliest epoch with maximum validation accuracy. Held-out
accuracies are reported after selection and do not choose the model.

New training is a separate, optional section. Its outputs go to
`results/models/tbdt_single_vg`, independently of the saved archive.


In [ ]:
# Run All displays archived results by default. Enable explicitly to fit new models.
RUN_TRAINING = False

# None uses all 24 configured Vg channels and all 24 parameter sets.
# Examples for a smaller new run: VG_INDICES = [0, 48, 92]; CANDIDATE_INDICES = [0, 16, 23].
VG_INDICES = None
CANDIDATE_INDICES = None
RESUME = True

# None uses CONFIG['OUTPUT']['models_dir']; otherwise provide a new output directory.
NEW_OUTPUT_ROOT = None


In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.Config_TBDT_Single_Vg import CONFIG as DEFAULT_CONFIG
from src import Single_vg_training as single_vg

CONFIG = deepcopy(DEFAULT_CONFIG)
ARCHIVE_ROOT = PROJECT_ROOT / "results" / "single_vg"
TRAINING_OUTPUT_ROOT = (
    Path(NEW_OUTPUT_ROOT).expanduser().resolve()
    if NEW_OUTPUT_ROOT is not None
    else PROJECT_ROOT / CONFIG["OUTPUT"]["models_dir"]
)
print("Archived results:", ARCHIVE_ROOT)
print("New training outputs:", TRAINING_OUTPUT_ROOT)


## The 24 parameter sets

The candidate order is fixed: hidden size, learning rate, weight decay, then dropout.
The grid is 3 × 2 × 2 × 2. Other architecture and training settings remain in the config.
Single-channel training disables Vg input dropout and uses the same CNN1D_TBDT architecture.


In [ ]:
candidate_table = pd.DataFrame(single_vg.expand_parameter_sets(CONFIG))
print(f"{len(CONFIG['VG_INDICES'])} Vg channels × {len(candidate_table)} candidates = "
      f"{len(CONFIG['VG_INDICES']) * len(candidate_table)} planned fits")
with pd.option_context("display.max_rows", len(candidate_table), "display.max_columns", None):
    display(candidate_table)


## Archived results

The following cells load the archived single-Vg results without retraining.
All accuracies are percentages.


In [ ]:
archived_summary = single_vg.load_single_vg_results(ARCHIVE_ROOT)
archived_candidates = pd.read_csv(ARCHIVE_ROOT / "completed_runs.csv")
print("ARCHIVED RESULTS — selected models:", len(archived_summary),
      "| completed candidates:", len(archived_candidates))
metric_columns = [
    "vg_index", "vg_value", "candidate_index", "selected_epoch",
    "validation_accuracy", "heldout_test_accuracy",
    "heldout_test_accuracy_rh0", "heldout_test_accuracy_rh30",
    "heldout_test_accuracy_rh50",
]
with pd.option_context("display.max_rows", len(archived_summary), "display.max_columns", None):
    display(archived_summary.loc[:, metric_columns])
display(
    archived_candidates.groupby("vg_index", sort=True)
    .size().rename("completed_candidates").to_frame()
)


In [ ]:
for metric, label in [
    ("rh0", "RH 0%"), ("rh30", "RH 30%"),
    ("rh50", "RH 50%"), ("overall", "all held-out RH conditions"),
]:
    fig, ax = single_vg.plot_single_vg_results(archived_summary, metric=metric)
    ax.set_title(f"Archived results — {label}")
    plt.show()


## Optional: train new models

Set `RUN_TRAINING = True` in the settings cell, then run the notebook again.
Use `VG_INDICES` and `CANDIDATE_INDICES` to limit the experiment before starting.
The full grid requests 576 fits, each with up to 200 epochs and patience 100.

Raw data are loaded only in the enabled branch below. The full 96-channel preprocessing
and train/validation partition are prepared once; the sweep then selects one cropped Vg
channel for each fit. Validation selects checkpoints and candidates. The external
held-out set supplies overall and per-RH reporting after selection.

`RESUME = True` reuses matching completed runs in the new output directory. The archived
results above remain available independently of new training.


In [ ]:
new_results = None
if RUN_TRAINING:
    from src import Data_pipeline

    chosen_vg_indices = CONFIG["VG_INDICES"] if VG_INDICES is None else VG_INDICES
    chosen_candidate_indices = (
        candidate_table["candidate_index"].tolist()
        if CANDIDATE_INDICES is None else CANDIDATE_INDICES
    )
    print("NEW TRAINING — requested fits:",
          len(chosen_vg_indices) * len(chosen_candidate_indices))
    selection = CONFIG["SELECTION"]
    preprocess = CONFIG["PREPROCESS"]
    bundle = Data_pipeline.prepare_data_bundle(
        data_root=Data_pipeline.resolve_data_root(CONFIG["DATASET"]["data_dir"]),
        train_rh_values=selection["train_rh_values"],
        heldout_test_rh_values=selection["heldout_test_rh_values"],
        average_mode=preprocess["average_mode"],
        cut_window=preprocess["cut_window"],
        val_size=selection["train_val_split"]["val_size"],
        random_state=selection["train_val_split"]["random_state"],
    )
    new_results = single_vg.run_single_vg_sweep(
        CONFIG, bundle,
        output_root=TRAINING_OUTPUT_ROOT,
        vg_indices=chosen_vg_indices,
        candidate_indices=chosen_candidate_indices,
        resume=RESUME,
    )
else:
    print("New training is disabled. Only saved archive results have been loaded.")


In [ ]:
if new_results is not None:
    new_summary = new_results["summary"]
    print("NEW TRAINING RESULTS:", new_results["output_root"])
    display(new_summary)
    fig, ax = single_vg.plot_single_vg_results(new_summary, metric="rh30")
    ax.set_title("New training - " + ax.get_title())
    plt.show()


## Reuse saved artifacts

`single_vg.load_single_vg_results(TRAINING_OUTPUT_ROOT)` reads a saved new sweep without
retraining. `single_vg.load_archived_model(vg_index, archive_root=ARCHIVE_ROOT, device="cpu")`
loads one selected archived checkpoint and its metadata. Use the archived predictions
and tables for analyses that do not require new inference.

The archive's `README.md` and `archive_manifest.json` document the saved
artifacts and checksums.
